# 单细胞 ATAC 测序

(染色质-可及性-介绍-关键-外卖-1)=
## 动机

生物体的每个细胞都具有相同的 DNA 以及称为基因的同一组功能单元。考虑到这一点，是什么决定了从免疫系统的自然杀伤细胞到在全身传递电化学信号的神经元的细胞的巨大多样性？在前面的章节中，我们看到可以从每个细胞的基因表达谱推断细胞身份和功能。基因表达的控制是由 DNA 甲基化、组蛋白修饰和转录因子活性等调控机制的复杂相互作用驱动的。 {term}`Chromatin`可访问性在很大程度上反映了细胞的综合调节状态，充当描述细胞身份的 {term}`mRNA <Messenger RNA (mRNA)>`级别的正交信息层。此外，探索染色质可及性概况可以进一步了解 scRNA-seq 数据可能无法捕获的基因调控机制和细胞分化过程。

:::{figure-md} Mechanisms influencing chromatin accessibility
<img src="../_static/images/sc_atac/mechanisms_overview.png" alt="Accessibility regulation" class="bg-primary mb-1" width="800px">

影响染色质可及性的机制概述。使用 BioRender.com 创建。
:::

如上所述，染色质可及性受到高阶结构直至低级 DNA 修饰的影响。 **(1)** 由支架/基质附着区 (S/MAR) 和核周边蛋白质（例如核孔复合物 (NPC) 或核纤层蛋白）驱动的染色质支架会影响染色质致密性和基因表达 {cite}`atac:narwade_mapping_2019, atac:buchwalter_coaching_2019`。 **(2, 3)** 与开放常染色质相比，通常称为密集异染色质的更多局部可及性可以通过 ATP 依赖性和 ATP 独立染色质重塑复合物以及组蛋白修饰（例如乙酰化、甲基化和磷酸化）主动控制。 **(4)** 转录因子的结合也可以影响核小体定位并导致组蛋白修饰酶和染色质重塑剂的募集。 **(5)** 在 DNA 水平上，{term}`CpG`位点的甲基化影响各种蛋白质的结合亲和力，包括转录因子和组蛋白修饰酶，这些蛋白质的结合导致相应基因组区域的沉默。对于动画可视化，我们还推荐有关表观遗传学和基因活性调节的 [this 2 minute video](https://www.youtube.com/watch?v=XelGO582s4U)（归功于伊利诺伊大学 SQE 的 Nicole Ethen）。有关基因组调控和 TF 活性的全面且最新的综述，我们参考{cite}`atac:isbel_generating_2022`。

总而言之，定义细胞身份的一个重要组成部分是每个细胞的调节状态。在本章中，我们重点关注通过 **Single-Cell Assay for Transposase-Accessible Chromatin 和 High-Throughput Sequencing (scATAC-seq)** 测量的染色质可及性数据，或作为 **10x 多组分析（scATAC 与 scRNA-seq 组合）** 的一部分。 

在引导您完成预处理步骤后，此分析将使我们能够：
1) 使用正交方法进行 scRNA-seq 分析来表征细胞身份
2) 识别细胞状态特异性转录调节因子
3）将基因表达与序列特征联系起来
4）解开驱动细胞分化和疾病状态的表观遗传机制


## 实验分析

目前，市售试剂盒是最广泛使用的实验方案，因此我们展示了对 10x 多组分析生成的数据的分析（稍作修改，这也适用于单峰 10x 单细胞 ATAC-seq 分析生成的数据）。

用于测量染色质可及性的关键原理是使用高-Throughput Sequencing 检测转座酶-Accessible Chromatin。 
起点是感兴趣组织的单细胞悬浮液。提取细胞核，并使用 Tn5 转座酶进行批量转座，该转座酶与染色质中的开放区域结合并生成标记的 DNA 片段。然后将细胞核加载到 10x Chromium Controller 上，并形成含有凝胶珠和单细胞的液滴，也称为 Gel Bead 乳液 (GEM)。在每个液滴内，RNA 分子和 DNA 片段都带有条形码，在溶解 GEM 后，预扩增核苷酸序列以接收最终的 scATAC-seq 和 scRNA-seq 文库。 

下图中，我们说明了scATAC-seq部分{cite}`atac:martens_modeling_2022`的分片过程。 scATAC-seq 使用 Tn5 转座酶将测序接头插入单细胞的开放染色质区域，从而导致 DNA 的裂解和测序接头的附着以创建 Tn5 片段。两次 Tn5 插入会产生一个带有测序接头的片段，并且插入方向至关重要，因为只有侧翼有两个不同接头的片段才能被捕获和扩增。然后对扩增的片段进行双端测序并与参考基因组进行比对。

:::{figure-md} ATAC-seq_概述
<img src="../_static/images/sc_atac/mechanisms_overview.png" alt="ATAC-seq overview" class="bg-primary mb-1" width="800px">
ATAC-seq 概述。图片来自 {cite}`atac:martens_modeling_2022`。
:::

(染色质-可及性-介绍-关键-外卖-2)=
## 数据特征——特征定义和稀疏性 

单细胞 ATAC-seq 数据测量整个基因组的染色质可及性。由于这包括编码区和非编码区，因此基因不能用作预定义特征，就像 scRNA-seq 数据的情况一样。相反，定义具有生物学意义的特征的最常见方法是检测与背景相比具有高可及性的区域，即沿基因组的片段计数分布的峰值。编码区中的峰表明基因可能被转录，而在非编码区中，可及性被视为结合转录因子等调节蛋白的先决条件或结果。然而，对数据集的所有细胞调用峰值可以隐藏细胞类型特定的可访问性或稀有细胞类型的可访问性概况。因此，提出的解决方案是调用特定于簇的峰值，这需要事先对细胞进行与峰值无关的聚类。 SnapATAC{cite}`atac:fang_comprehensive_2021`和 ArchR{cite}`atac:granja_archr_2021`提出了一种分箱策略，该策略通过将整个基因组划分为大小均匀的窗口并使用此特征集对细胞进行聚类来创建特征。 

一旦以一种或另一种方式定义了特征集，就为每个细胞定义了这些特征中 Tn5 活性的测量。使用三种主要方法：计数与特征重叠的读数、计数与特征重叠的片段以及二值化。虽然 10x Genomics Cell Ranger ATAC 管道计数读取重叠的峰区域，但广泛使用的 Signac 框架 {cite}`atac:stuart_multimodal_2020`计算与特征重叠的片段数量。另一方面，ArchR {cite}`atac:granja_archr_2021`默认计算读取结束并对其进行二值化。

值得注意的是，计数读数和计数片段之间存在一些差异。在 scATAC-seq 中，双端测序生成两个通常彼此非常接近的读数。因此，仅当一对读取位于特征 {cite}`atac:martens_modeling_2022`之外时，才会生成不均匀计数。这意味着所使用的计数策略会影响最终的计数分布。已经表明，读取计数策略导致计数分布的均匀计数多于不均匀计数，而计数片段则没有这种效果{cite}`atac:martens_modeling_2022, atac:miao_is_2022`。

scATAC-seq 数据的另一个重要特征是其高度稀疏性。由于二倍体生物体的每个细胞中只有两个 DNA 副本，因此给定碱基对位置的最大计数数为 2（请注意，峰或箱中可以有两个以上的计数，因为范围很长）。这可能导致许多特征的计数为零，从而导致计数矩阵高度稀疏。考虑到这一点，一些方法采用对计数进行二值化的方法，这意味着一旦一次读取或片段与 {cite}`atac:granja_archr_2021, atac:bravo_gonzalez-blas_cistopic_2019, atac:bravo_gonzalez-blas_cistopic_2019, atac:ashuach_peakvi_2022`重叠，该功能就被称为可访问。然而，二值化可能会导致信息丢失，并且在检测可访问性的微小差异 {cite}`atac:martens_modeling_2022`时可能不太敏感。

值得注意的是，scATAC-seq 数据的最佳计数策略仍在争论中，需要进一步的独立基准测试。最终，计数策略的选择将取决于具体的研究问题和数据集的特征。

## 数据分析工作流程概述

在以下部分中，我们将指导您完成分析 scATAC-seq 数据的标准工作流程。随附的概述图显示了分析的各个阶段，并强调了用于此目的的流行框架之间的差异。首先，我们将使用 Python 和 muon 解释质量控制和降维的概念。最后，我们将演示如何将 muon 对象传输到 R 并使用 Signac 执行数据可视化和解释。 

:::{figure-md} ATAC-seq_overview
<img src="../_static/images/sc_atac/overview_atac.jpeg" alt="ATAC-seq overview" class="bg-primary mb-1" width="900px">

scATAC-seq 分析步骤概述。 
:::

## 参考

```{bibliography}
:filter: docname in docnames
:labelprefix: atac
```

## 贡献者

我们衷心感谢以下人员的贡献：

### 作者

* Christopher Lance
* Laura Martens

### 审稿人

* Lukas Heumos